# Scenario 2: Code Generation with Claude Code

**Domains tested:** Claude Code Config (D3 -- 20%), Context Management (D5 -- 15%)

---

## Scenario Context

You are the engineering lead at **CodeCraft**, a SaaS company with 40 developers across 3 teams (frontend, backend, platform). Your organization has adopted Claude Code as the primary AI coding assistant. You need to configure it for consistent, team-aligned code generation across the entire codebase.

**Production requirements:**
- Claude Code must follow team-specific coding conventions (React/TypeScript for frontend, Go for backend, Python for platform)
- Architectural changes require plan mode review before execution
- Path-specific rules must enforce different behaviors for tests, migrations, and API endpoints
- Custom slash commands (skills) should be available for common workflows
- The configuration must be debuggable when rules don't apply as expected

**Repository structure:**
```
codecraft/
  CLAUDE.md                 # root config
  frontend/
    CLAUDE.md               # frontend team config
    src/components/
    src/hooks/
  backend/
    CLAUDE.md               # backend team config
    cmd/api/
    internal/
  platform/
    CLAUDE.md               # platform team config
    scripts/
    infra/
  .claude/
    rules/
      testing.md            # rules for test files
      migrations.md         # rules for DB migrations
      api-endpoints.md      # rules for API handlers
    skills/
      review-pr.md          # PR review skill
      generate-tests.md     # test generation skill
```

---

In [ ]:
import anthropic
import json
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-20250514"

---

## Part 1: Exam-Format Questions

---

### Question 1 (D3)

A developer on the frontend team reports that Claude Code is generating Go-style error handling in their React components. They have a `frontend/CLAUDE.md` that specifies TypeScript conventions, but it is being ignored. The root `CLAUDE.md` contains the instruction: "Use Go error handling patterns (return error as second value)."

What is the most likely cause?

- **A)** Child `CLAUDE.md` files do not override parent `CLAUDE.md` files -- the root always takes precedence
- **B)** `CLAUDE.md` files cannot contain language-specific instructions
- **C)** The `frontend/CLAUDE.md` file has a syntax error and Claude Code silently ignores it
- **D)** The developer is working from the repo root, so Claude Code loads the root `CLAUDE.md` but the `frontend/CLAUDE.md` only loads when the working directory is inside `frontend/`

In [ ]:
q1 = {
    "correct": "D",
    "explanation": (
        "CLAUDE.md files form a hierarchy. The root CLAUDE.md always loads. Child CLAUDE.md files "
        "only load when the working context includes files from that directory. If the developer "
        "is running Claude Code from the repo root and the conversation hasn't touched any files "
        "in frontend/, the frontend/CLAUDE.md won't be loaded. Option A is wrong -- child files "
        "DO add to (and can effectively override) parent instructions. Option B is wrong -- CLAUDE.md "
        "can contain any instructions. Option C is plausible but the most common cause is the "
        "directory context issue. This tests Domain 3: understanding the CLAUDE.md hierarchy and "
        "when each level is active."
    )
}
print(f"Correct: {q1['correct']}")
print(f"\nExplanation: {q1['explanation']}")

### Question 2 (D3)

You want different Claude Code rules for test files (`*_test.go`, `*.test.ts`) vs production code. Where should these rules be configured?

- **A)** In `.claude/rules/testing.md` with a YAML frontmatter `paths:` glob that matches test file patterns
- **B)** In the root `CLAUDE.md` with if/else logic: "If the file ends in `_test.go`, do X, otherwise do Y"
- **C)** In separate `CLAUDE.md` files in each test directory
- **D)** In the system prompt when starting Claude Code with `--system-prompt`

In [ ]:
q2 = {
    "correct": "A",
    "explanation": (
        ".claude/rules/ files with YAML frontmatter paths: globs are the purpose-built mechanism "
        "for path-specific rules. Example:\n\n"
        "---\npaths:\n  - '**/*_test.go'\n  - '**/*.test.ts'\n---\n"
        "Rules here only apply to test files.\n\n"
        "Option B puts conditional logic in the prompt, which is fragile and hard to maintain. "
        "Option C would require CLAUDE.md files in every test directory, which doesn't scale. "
        "Option D is not a real Claude Code flag. This tests Domain 3: .claude/rules/ with "
        "glob patterns is the correct mechanism for path-conditional behavior."
    )
}
print(f"Correct: {q2['correct']}")
print(f"\nExplanation: {q2['explanation']}")

### Question 3 (D3 + D5)

You create a custom skill at `.claude/skills/generate-tests.md` that generates unit tests for a given file. The skill needs its own isolated context to avoid polluting the main conversation. How should you configure this?

- **A)** Add `context: isolated` to prevent any context sharing
- **B)** Clear the conversation history before invoking the skill
- **C)** Add `context: fork` to the skill's YAML frontmatter so it runs in a forked context that doesn't affect the parent conversation
- **D)** Use a separate Claude Code session for skills

In [ ]:
q3 = {
    "correct": "C",
    "explanation": (
        "context: fork is the correct mechanism for skills that need isolated context. A forked "
        "context inherits the current conversation state but runs independently -- its tool calls "
        "and outputs don't pollute the parent conversation. This is critical for skills that "
        "perform multi-step operations (like generating tests) where intermediate steps would "
        "clutter the main conversation. Option A uses a non-existent setting. Option B destroys "
        "useful context. Option D is impractical. This tests Domain 3 (skill configuration) and "
        "Domain 5 (context management through isolation)."
    )
}
print(f"Correct: {q3['correct']}")
print(f"\nExplanation: {q3['explanation']}")

### Question 4 (D3)

A junior developer proposes a major refactoring of the authentication module. You want Claude Code to first create a plan for review before making any changes. What is the correct approach?

- **A)** Add "Do not make changes without asking first" to `CLAUDE.md`
- **B)** Use plan mode (`Shift+Tab` to toggle) which forces Claude to outline its approach before executing, then review and approve the plan before switching to act mode
- **C)** Use `--dry-run` flag to preview changes
- **D)** Set `max_tokens` very low so Claude can only describe what it would do

In [ ]:
q4 = {
    "correct": "B",
    "explanation": (
        "Plan mode is the built-in Claude Code feature for exactly this use case. In plan mode, "
        "Claude analyzes the codebase and produces a structured plan without making any changes. "
        "You review the plan, provide feedback, and only switch to act mode when satisfied. "
        "Option A is a prompt-level instruction that the model might not follow consistently. "
        "Option C is not a real Claude Code flag. Option D is a hack that would produce "
        "truncated, unusable output. This tests Domain 3: knowing when and how to use plan mode "
        "for architectural changes that need human review."
    )
}
print(f"Correct: {q4['correct']}")
print(f"\nExplanation: {q4['explanation']}")

### Question 5 (D3 + D5)

Your team has rules in `.claude/rules/api-endpoints.md` with `paths: ['backend/cmd/api/**/*.go']`. A developer reports the rules aren't applying when they ask Claude Code to "create a new API endpoint." Investigation shows Claude Code is creating the file at `backend/internal/handlers/new_endpoint.go`.

What is happening?

- **A)** The file is being created at a path that does not match the glob pattern `backend/cmd/api/**/*.go` -- the rules only apply to files under `backend/cmd/api/`, not `backend/internal/handlers/`
- **B)** The glob pattern is wrong -- `**/*.go` doesn't match nested directories
- **C)** The rules file has invalid YAML frontmatter
- **D)** `.claude/rules/` files cannot have path restrictions

In [ ]:
q5 = {
    "correct": "A",
    "explanation": (
        "The glob pattern backend/cmd/api/**/*.go only matches files under the backend/cmd/api/ "
        "directory. If Claude Code creates the file at backend/internal/handlers/, the pattern "
        "does not match and the rules don't apply. The fix is either: (1) update the glob to "
        "include the internal/handlers/ path, or (2) instruct Claude Code where to place API "
        "handlers so they match the pattern. Option B is wrong -- **/ does match nested dirs. "
        "Option C is possible but less likely. Option D is wrong -- path restrictions are a "
        "core feature. This tests Domain 3: understanding that glob patterns in .claude/rules/ "
        "are path-sensitive and must match the actual file locations."
    )
}
print(f"Correct: {q5['correct']}")
print(f"\nExplanation: {q5['explanation']}")

### Question 6 (D3 + D5)

After a long coding session (50+ interactions), developers report that Claude Code "forgets" instructions from `CLAUDE.md`. Early interactions follow the rules, but later ones drift. What is the most likely explanation?

- **A)** `CLAUDE.md` is only loaded once at the start of the session and expires after a timeout
- **B)** Claude Code has a bug that drops system context after 50 interactions
- **C)** As the conversation grows, the `CLAUDE.md` instructions in the system prompt are proportionally less prominent compared to the accumulated conversation context, leading to attention degradation
- **D)** The model's context window is full and `CLAUDE.md` content is being truncated

In [ ]:
q6 = {
    "correct": "C",
    "explanation": (
        "This is a context management issue (Domain 5) manifesting in the Claude Code config "
        "domain (D3). As conversation history grows, the system prompt (which contains CLAUDE.md "
        "instructions) becomes a smaller proportion of the total context. The model's attention "
        "to system instructions can degrade when overwhelmed by conversation turns. Mitigations: "
        "(1) start fresh sessions for new tasks, (2) use /clear to reset conversation context, "
        "(3) use compact to summarize older context. Option A is wrong -- CLAUDE.md is included "
        "with every API call. Option B is speculative. Option D is possible but unlikely to be "
        "the primary issue with modern context windows."
    )
}
print(f"Correct: {q6['correct']}")
print(f"\nExplanation: {q6['explanation']}")

---

## Part 2: Hands-On Build

Build the CLAUDE.md hierarchy and rules configuration described in the scenario. We will simulate the configuration and test it.

---

### Step 1: Define the CLAUDE.md hierarchy

In [ ]:
# Simulate the CLAUDE.md hierarchy as a data structure

CLAUDE_MD_HIERARCHY = {
    "root": {
        "path": "codecraft/CLAUDE.md",
        "content": """
# CodeCraft Development Guidelines

## General Rules
- All code must have tests before merging
- Use meaningful variable names (no single-letter variables except loop counters)
- All API endpoints must have OpenAPI documentation
- Error messages must be user-facing friendly, log technical details separately

## Architecture
- Frontend: React/TypeScript (see frontend/CLAUDE.md)
- Backend: Go (see backend/CLAUDE.md)
- Platform: Python (see platform/CLAUDE.md)

## Commit Conventions
- Use conventional commits: feat:, fix:, refactor:, test:, docs:
- One logical change per commit
"""
    },
    "frontend": {
        "path": "codecraft/frontend/CLAUDE.md",
        "content": """
# Frontend Team Guidelines

## Language & Framework
- TypeScript strict mode -- no `any` types
- React functional components with hooks (no class components)
- Use React Query for server state, Zustand for client state

## Error Handling
- Use Error Boundaries for component errors
- Use try/catch with typed error responses for API calls
- Never use Go-style error returns (this is TypeScript, use exceptions)

## Testing
- Jest + React Testing Library
- Test behavior, not implementation
- Mock API calls with MSW (Mock Service Worker)
"""
    },
    "backend": {
        "path": "codecraft/backend/CLAUDE.md",
        "content": """
# Backend Team Guidelines

## Language & Style
- Go 1.22+ with standard library preferred over third-party packages
- Use Go error handling patterns (return error as second value)
- Use context.Context for cancellation and timeouts

## API Design
- RESTful endpoints under /api/v1/
- Use middleware for auth, logging, rate limiting
- Return structured JSON errors: {"error": {"code": "...", "message": "..."}}

## Testing
- Table-driven tests with descriptive subtests
- Use testify for assertions
- Integration tests use testcontainers
"""
    },
    "platform": {
        "path": "codecraft/platform/CLAUDE.md",
        "content": """
# Platform Team Guidelines

## Language & Style
- Python 3.12+ with type hints on all function signatures
- Use Pydantic for data validation
- Use asyncio for I/O-bound operations

## Infrastructure
- Terraform for infrastructure-as-code
- Docker Compose for local development
- All scripts must be idempotent

## Testing
- pytest with fixtures
- Use hypothesis for property-based testing where applicable
"""
    }
}

print("CLAUDE.md hierarchy defined:")
for level, config in CLAUDE_MD_HIERARCHY.items():
    print(f"  {level}: {config['path']}")

### Step 2: Define path-specific rules

In [ ]:
RULES = {
    "testing": {
        "path": ".claude/rules/testing.md",
        "frontmatter": {
            "paths": ["**/*_test.go", "**/*.test.ts", "**/*.test.tsx", "**/test_*.py"]
        },
        "content": """
## Test File Rules
- Every test must have a descriptive name: test_<behavior>_<condition>_<expected>
- No test should depend on another test's state
- Use factories/fixtures for test data, never hardcode
- Mock external services, never call real APIs in tests
- Each test file tests exactly one module/component
"""
    },
    "migrations": {
        "path": ".claude/rules/migrations.md",
        "frontmatter": {
            "paths": ["**/migrations/**/*.sql", "**/migrations/**/*.go"]
        },
        "content": """
## Database Migration Rules
- Every migration must be reversible (include both up and down)
- Never drop columns in production -- mark as deprecated first
- Use transactions for all schema changes
- Add indexes CONCURRENTLY to avoid locking
- Test migrations against a copy of production data
"""
    },
    "api_endpoints": {
        "path": ".claude/rules/api-endpoints.md",
        "frontmatter": {
            "paths": ["backend/cmd/api/**/*.go", "backend/internal/handlers/**/*.go"]
        },
        "content": """
## API Endpoint Rules
- Every endpoint must validate input with struct tags
- Return appropriate HTTP status codes (don't return 200 for errors)
- Rate limit all public endpoints
- Log request ID, method, path, status, and latency
- All responses must include Content-Type: application/json
"""
    }
}

print("Path-specific rules defined:")
for name, rule in RULES.items():
    print(f"  {name}: {rule['frontmatter']['paths']}")

### Step 3: Simulate CLAUDE.md resolution

In [ ]:
import fnmatch


def resolve_claude_md(file_path: str) -> dict:
    """Simulate CLAUDE.md resolution for a given file path.
    Returns which CLAUDE.md files and rules would be active."""

    active_configs = []

    # Root CLAUDE.md always loads
    active_configs.append({
        "source": "root CLAUDE.md",
        "path": CLAUDE_MD_HIERARCHY["root"]["path"]
    })

    # Check which child CLAUDE.md applies
    for level in ["frontend", "backend", "platform"]:
        if file_path.startswith(f"codecraft/{level}/"):
            active_configs.append({
                "source": f"{level} CLAUDE.md",
                "path": CLAUDE_MD_HIERARCHY[level]["path"]
            })

    # Check which rules apply
    active_rules = []
    for name, rule in RULES.items():
        for pattern in rule["frontmatter"]["paths"]:
            # Strip the codecraft/ prefix for matching
            relative_path = file_path.replace("codecraft/", "", 1)
            if fnmatch.fnmatch(relative_path, pattern):
                active_rules.append({
                    "rule": name,
                    "matched_pattern": pattern,
                    "path": rule["path"]
                })
                break

    return {"configs": active_configs, "rules": active_rules}


# Test resolution for different file paths
test_paths = [
    "codecraft/frontend/src/components/Button.tsx",
    "codecraft/frontend/src/components/Button.test.tsx",
    "codecraft/backend/cmd/api/handlers/orders.go",
    "codecraft/backend/cmd/api/handlers/orders_test.go",
    "codecraft/backend/internal/handlers/auth.go",
    "codecraft/platform/scripts/deploy.py",
    "codecraft/backend/migrations/001_create_users.sql",
]

print("CLAUDE.md Resolution Test:\n")
for path in test_paths:
    result = resolve_claude_md(path)
    configs = [c["source"] for c in result["configs"]]
    rules = [f"{r['rule']} ({r['matched_pattern']})" for r in result["rules"]]
    print(f"  {path}")
    print(f"    Configs: {', '.join(configs)}")
    print(f"    Rules:   {', '.join(rules) if rules else 'none'}")
    print()

### Step 4: Test rule application with Claude

In [ ]:
def generate_with_rules(task: str, file_path: str) -> str:
    """Simulate Claude Code generation with resolved CLAUDE.md and rules."""
    resolution = resolve_claude_md(file_path)

    # Build the system prompt from active configs and rules
    system_parts = ["You are Claude Code, an AI coding assistant. Follow these rules:\n"]

    for config in resolution["configs"]:
        for level, data in CLAUDE_MD_HIERARCHY.items():
            if data["path"] == config["path"]:
                system_parts.append(f"--- {config['source']} ---")
                system_parts.append(data["content"])

    for rule in resolution["rules"]:
        for name, data in RULES.items():
            if name == rule["rule"]:
                system_parts.append(f"--- Rule: {name} (matched {rule['matched_pattern']}) ---")
                system_parts.append(data["content"])

    system_prompt = "\n".join(system_parts)

    response = client.messages.create(
        model=MODEL,
        max_tokens=2048,
        system=system_prompt,
        messages=[{"role": "user", "content": f"Task: {task}\nFile: {file_path}\n\nGenerate the code."}]
    )

    return response.content[0].text


# Test: Generate a React component (should use TypeScript, not Go error handling)
print("=== Test: Frontend Component ===")
print("File: codecraft/frontend/src/components/UserProfile.tsx\n")

result = generate_with_rules(
    "Create a UserProfile component that fetches and displays user data from /api/v1/users/:id",
    "codecraft/frontend/src/components/UserProfile.tsx"
)
print(result[:1500])

In [ ]:
# Test: Generate a Go test file (should follow both backend + testing rules)
print("=== Test: Backend Test File ===")
print("File: codecraft/backend/cmd/api/handlers/orders_test.go\n")

result = generate_with_rules(
    "Create tests for the orders API handler. Test listing orders, creating an order, and handling invalid input.",
    "codecraft/backend/cmd/api/handlers/orders_test.go"
)
print(result[:1500])

---

## Part 3: Failure Injections

---

### Failure 1: Missing child CLAUDE.md (wrong language conventions)

In [ ]:
print("=== FAILURE INJECTION: No frontend CLAUDE.md ===")
print("What happens when we generate frontend code with only the root CLAUDE.md?\n")

# Simulate: only root CLAUDE.md (no frontend override)
system_prompt_root_only = (
    "You are Claude Code. Follow these rules:\n"
    + CLAUDE_MD_HIERARCHY["root"]["content"]
    + "\n\nFor Go backend code, use Go error handling patterns (return error as second value)."
)

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=system_prompt_root_only,
    messages=[{
        "role": "user",
        "content": "Create a UserProfile React component that fetches user data with error handling."
    }]
)

result = response.content[0].text
print(result[:1000])

print("\n--- DIAGNOSIS ---")
print("Without the frontend/CLAUDE.md, the model may apply Go-style error patterns")
print("to TypeScript code. The root CLAUDE.md mentions Go error handling, and without")
print("the frontend override, there's no instruction to use TypeScript exceptions instead.")
print("LESSON: Each team's CLAUDE.md must explicitly state its conventions (Domain 3).")

### Failure 2: Glob pattern mismatch

In [ ]:
print("=== FAILURE INJECTION: Glob pattern doesn't match ===")
print("Testing a file at an unexpected path...\n")

# This file is at a path not covered by the api-endpoints rule
unexpected_path = "codecraft/backend/pkg/api/router.go"
resolution = resolve_claude_md(unexpected_path)

print(f"File: {unexpected_path}")
print(f"Active configs: {[c['source'] for c in resolution['configs']]}")
print(f"Active rules: {[r['rule'] for r in resolution['rules']]}")

print("\n--- DIAGNOSIS ---")
print("The api-endpoints rule has paths: ['backend/cmd/api/**/*.go', 'backend/internal/handlers/**/*.go']")
print(f"But {unexpected_path} is under backend/pkg/api/ -- not matched!")
print("")
print("Fix options:")
print("  1. Add 'backend/pkg/api/**/*.go' to the api-endpoints rule paths")
print("  2. Restructure the codebase to keep API code in matched directories")
print("  3. Use a broader glob like 'backend/**/api/**/*.go'")
print("")
print("LESSON: Regularly audit .claude/rules/ globs against actual file paths (Domain 3).")

### Failure 3: Long session context drift

In [ ]:
print("=== FAILURE INJECTION: Context drift in long sessions ===")
print("Simulating attention degradation with growing conversation context.\n")

# Build a system prompt with CLAUDE.md rules
claude_md_instructions = CLAUDE_MD_HIERARCHY["backend"]["content"]

# Simulate a short conversation (rules should be followed)
short_messages = [
    {"role": "user", "content": "Create a function to validate email addresses in Go."}
]

response_short = client.messages.create(
    model=MODEL,
    max_tokens=512,
    system=f"Follow these rules:\n{claude_md_instructions}",
    messages=short_messages
)
print("Short session (1 turn):")
print(response_short.content[0].text[:500])

# Simulate a long conversation (50+ turns of padding)
long_messages = []
for i in range(25):
    long_messages.append({"role": "user", "content": f"Explain concept {i} about Go channels and goroutines in detail."})
    long_messages.append({"role": "assistant", "content": f"Here is a detailed explanation of concept {i} about Go concurrency patterns. Go channels provide a way to communicate between goroutines safely. " * 10})

# Now add the real request at the end
long_messages.append({"role": "user", "content": "Create a function to validate email addresses in Go."})

response_long = client.messages.create(
    model=MODEL,
    max_tokens=512,
    system=f"Follow these rules:\n{claude_md_instructions}",
    messages=long_messages
)
print("\nLong session (50+ turns):")
print(response_long.content[0].text[:500])

print("\n--- DIAGNOSIS ---")
print("Compare the two outputs. In long sessions, the model may:")
print("  - Forget specific CLAUDE.md conventions")
print("  - Use different code patterns than in the short session")
print("  - Miss test requirements or style rules")
print("")
print("Mitigation: Use /clear between tasks, or /compact to summarize old context.")
print("LESSON: Context window size != context quality (Domain 5).")

---

## Key Takeaways

| Concept | Domain | Lesson |
|---------|--------|--------|
| CLAUDE.md hierarchy | D3 | Root always loads; child loads only when its directory is in context |
| .claude/rules/ globs | D3 | Path-specific rules activate only when file paths match the glob pattern |
| context: fork for skills | D3, D5 | Skills needing isolated context use fork to avoid polluting the main conversation |
| Plan mode | D3 | Use for architectural changes that need human review before execution |
| Context drift | D5 | Long sessions degrade attention to system instructions -- use /clear or /compact |
| Glob debugging | D3 | When rules don't apply, check that file paths actually match the glob patterns |